# 🎬 YouTube RAG Chatbot — Day 11
> **Stack:** LangChain · FAISS · HuggingFace Embeddings · Groq (free LLM API)
>
> Paste any YouTube URL → ask questions → get answers grounded in the actual transcript.

---

## 📦 Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install langchain langchain-community langchain-groq \
             youtube-transcript-api faiss-cpu \
             sentence-transformers tiktoken \
             groq --quiet

## 🔑 Step 2 — Set Your API Key

We use **Groq** — it's **free** and extremely fast (uses LLaMA 3 under the hood).

👉 Get your free key at [console.groq.com](https://console.groq.com) (takes 30 seconds)

In [ ]:
import os
from google.colab import userdata

# Option A: Store in Colab Secrets (recommended — click the 🔑 key icon on the left)
# Then add a secret named GROQ_API_KEY
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Option B: Paste directly (less secure)
    os.environ["GROQ_API_KEY"] = "gsk_YOUR_KEY_HERE"  # ← replace this
    print("⚠️  Using hardcoded key — prefer Colab Secrets for security")

## 🔧 Step 3 — Import All Libraries

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse, parse_qs

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

print("✅ All libraries imported successfully")

## 🎥 Step 4 — Extract YouTube Transcript

Paste any YouTube URL below. Works with:
- `https://www.youtube.com/watch?v=VIDEO_ID`
- `https://youtu.be/VIDEO_ID`

In [ ]:
def extract_video_id(url: str) -> str:
    """Extract video ID from any YouTube URL format."""
    parsed = urlparse(url)
    if parsed.hostname in ('youtu.be',):
        return parsed.path[1:]
    if parsed.hostname in ('www.youtube.com', 'youtube.com'):
        return parse_qs(parsed.query).get('v', [None])[0]
    raise ValueError(f"Unrecognised YouTube URL: {url}")


def get_transcript(youtube_url: str) -> str:
    """Fetch transcript text from a YouTube video."""
    video_id = extract_video_id(youtube_url)
    print(f"📹 Fetching transcript for video ID: {video_id}")
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
    full_text = " ".join([entry["text"] for entry in transcript_list])
    print(f"✅ Transcript fetched — {len(full_text):,} characters")
    return full_text


# ✏️  Paste your YouTube URL here
YOUTUBE_URL = "https://www.youtube.com/watch?v=LhnCsygAvzY"   # CampusX LangChain RAG video

raw_transcript = get_transcript(YOUTUBE_URL)
print("\n📄 Preview (first 500 chars):")
print(raw_transcript[:500])

## ✂️ Step 5 — Chunk the Transcript

We split the transcript into overlapping chunks so the retriever can find the most relevant passage for any question.

In [ ]:
def chunk_transcript(text: str, chunk_size: int = 1000, chunk_overlap: int = 200):
    """Split transcript into overlapping chunks for better retrieval."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    docs = splitter.create_documents([text])
    print(f"✅ Created {len(docs)} chunks  "
          f"(size={chunk_size}, overlap={chunk_overlap})")
    return docs


chunks = chunk_transcript(raw_transcript)
print(f"\n🔍 Sample chunk:\n{chunks[0].page_content[:300]}...")

## 🧠 Step 6 — Create Vector Store (FAISS)

We embed each chunk using a HuggingFace model and store them in a FAISS index so we can do lightning-fast semantic search.

In [ ]:
print("⏳ Loading embedding model (downloads once, ~90 MB)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("⏳ Building FAISS vector store...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store ready — {vectorstore.index.ntotal} vectors indexed")

## 🤖 Step 7 — Build the RAG Chain

This wires together:
1. **Retriever** → finds the top-k relevant chunks from FAISS
2. **Prompt** → instructs the LLM to answer ONLY from retrieved context
3. **LLM** → Groq's LLaMA 3.1 (free, fast, great quality)

In [ ]:
# --- Retriever ---
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}   # fetch top 4 chunks
)

# --- Prompt template ---
PROMPT_TEMPLATE = """
You are a helpful assistant answering questions about a YouTube video.
Use ONLY the context below to answer. If the answer is not in the context,
say "I couldn't find that in the video transcript."

Context:
{context}

Question: {question}

Answer (be concise and accurate):
"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

# --- LLM (Groq — free tier) ---
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",   # fast & free
    temperature=0.2,
    max_tokens=512
)

# --- RAG Chain ---
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("✅ RAG chain is ready — let's ask some questions!")

## 💬 Step 8 — Ask Questions!

Run the cell below as many times as you like with different questions.

In [ ]:
def ask(question: str, show_sources: bool = False):
    """Query the YouTube RAG chatbot."""
    print(f"\n❓ Question: {question}")
    print("-" * 60)
    result = rag_chain.invoke({"query": question})
    print(f"🤖 Answer:\n{result['result']}")
    if show_sources:
        print("\n📚 Source chunks used:")
        for i, doc in enumerate(result["source_documents"], 1):
            print(f"  [{i}] {doc.page_content[:150]}...")
    print("-" * 60)
    return result


# ✏️  Change these questions to anything about your video
ask("What is the main topic of this video?")
ask("What tools or libraries are used?")
ask("What is RAG and why is it useful?", show_sources=True)

## 🖥️ Step 9 — Interactive Chat Mode

Run this cell for a simple terminal-style chat loop. Type `quit` to exit.

In [ ]:
print("🎬 YouTube RAG Chatbot — Interactive Mode")
print("Type your questions below. Enter 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("👋 Exiting chat. Great work today!")
        break
    ask(user_input)

## 💾 Step 10 — Save & Load Vector Store (Optional)

Avoid re-embedding every run by persisting the FAISS index to disk.

In [ ]:
# Save
vectorstore.save_local("yt_faiss_index")
print("✅ Vector store saved to ./yt_faiss_index")

# Load (run this in future sessions instead of rebuilding)
# loaded_vs = FAISS.load_local("yt_faiss_index", embeddings,
#                               allow_dangerous_deserialization=True)
# print("✅ Vector store loaded from disk")

## 📊 Step 11 — Visualise the Architecture

A quick summary of what we built today.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║            YouTube RAG Chatbot — Architecture            ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  YouTube URL                                             ║
║      │                                                   ║
║      ▼                                                   ║
║  YouTubeTranscriptAPI  ──►  Raw Transcript Text          ║
║                                      │                   ║
║                                      ▼                   ║
║                           RecursiveCharacterTextSplitter  ║
║                                      │                   ║
║                                      ▼                   ║
║                           HuggingFace Embeddings         ║
║                           (all-MiniLM-L6-v2)            ║
║                                      │                   ║
║                                      ▼                   ║
║                            FAISS Vector Store            ║
║                                      │                   ║
║  User Question  ────────► Retriever (top-k=4)           ║
║                                      │                   ║
║                                      ▼                   ║
║                           Prompt Template                ║
║                           (context + question)           ║
║                                      │                   ║
║                                      ▼                   ║
║                           Groq LLM (LLaMA 3.1)          ║
║                                      │                   ║
║                                      ▼                   ║
║                           Grounded Answer ✅             ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")

---
## ✅ Day 11 Complete!

**What you built:**
- Extracted a YouTube transcript programmatically
- Chunked it for semantic retrieval
- Created a FAISS vector index with HuggingFace embeddings
- Wired a RAG chain using LangChain + Groq (free LLM)
- Built an interactive chatbot grounded in the video content

**Tags:** `#RAG` `#LangChain` `#LLMs` `#Day11` `#100DaysOfCode` `#BuildInPublic`

---
*Built on Day 11 of the 100 Days of AI challenge 🚀*